# Schema-Aware NL2SQL — Fine-Tune on Spider (one notebook)

Runs the whole pipeline on a **free Colab T4 GPU**: clone → install → get Spider data → smoke test → LoRA fine-tune T5 → execution-accuracy eval → push merged model to the Hugging Face Hub.

**Before you start:**
1. `Runtime → Change runtime type → T4 GPU`.
2. Have a **Hugging Face write token** ready (huggingface.co/settings/tokens).
3. Download **`spider.zip`** once from the official Spider release (Yale LILY / Spider 1.0). You'll upload it in Step 3 (or set a Google Drive file id to auto-download).

## Step 0 — Confirm the GPU

In [ ]:
!nvidia-smi

## Step 1 — Settings (edit these)

In [ ]:
REPO_BRANCH = "feature/schema-aware-anydb"
BASE_MODEL  = "t5-base"          # use "t5-large" for best accuracy (slower, more VRAM)
EPOCHS      = 5
MODEL_REPO  = "Srijan-Ratrey/nl2sql-t5-spider"   # where the merged model gets pushed

# Optional: Google Drive file id for spider.zip to auto-download.
# Leave "" to upload spider.zip manually in Step 3.
SPIDER_GDRIVE_ID = ""
print("base:", BASE_MODEL, "| epochs:", EPOCHS, "| push to:", MODEL_REPO)

## Step 2 — Clone the repo & install dependencies

In [ ]:
!git clone -b {REPO_BRANCH} https://github.com/Srijan-Ratrey/Schema-Aware-Natural-Language-to-SQL-Agent.git
%cd Schema-Aware-Natural-Language-to-SQL-Agent
!pip -q install transformers datasets peft accelerate evaluate sqlglot sentencepiece
# Spider's HF loader may require this flag on newer `datasets` versions:
%env HF_DATASETS_TRUST_REMOTE_CODE=1

## Step 3 — Get the Spider data (schemas + databases)

Needs `tables.json` (schemas) and `database/` (the SQLite DBs, for execution-accuracy eval). Upload `spider.zip` when prompted, or set `SPIDER_GDRIVE_ID` above to auto-download.

In [ ]:
import os, glob, zipfile

if SPIDER_GDRIVE_ID:
    !pip -q install gdown
    !gdown {SPIDER_GDRIVE_ID} -O spider.zip
else:
    from google.colab import files
    print("Upload spider.zip (downloaded from the official Spider release)...")
    files.upload()

for z in glob.glob("spider*.zip"):
    print("Extracting", z)
    with zipfile.ZipFile(z) as f:
        f.extractall("spider_data")

TABLES_JSON = next(iter(glob.glob("spider_data/**/tables.json", recursive=True)), None)
_db_dirs = [d for d in glob.glob("spider_data/**/database", recursive=True) if os.path.isdir(d)]
DB_DIR = _db_dirs[0] if _db_dirs else None
print("TABLES_JSON:", TABLES_JSON)
print("DB_DIR:", DB_DIR)
assert TABLES_JSON and DB_DIR, "Could not find tables.json / database/ inside the zip."

## Step 4 — Smoke test (~2 min)
Validates the whole pipeline on a tiny sample before committing hours of compute.

In [ ]:
!python scripts/train_lora.py \
  --base-model t5-base \
  --tables-json "{TABLES_JSON}" \
  --epochs 1 --max-train-samples 200 \
  --output-dir smoke-test
print("\nSmoke test OK if it saved to smoke-test/ without errors.")

## Step 5 — Full fine-tune
Saves the LoRA adapter to `nl2sql-t5-lora/` (checkpoint each epoch). ~2–4h on a T4 for `t5-large`. If you hit OOM, re-run with `--batch-size 2 --grad-accum 16` or use `t5-base`.

In [ ]:
!python scripts/train_lora.py \
  --base-model "{BASE_MODEL}" \
  --tables-json "{TABLES_JSON}" \
  --epochs {EPOCHS} \
  --output-dir nl2sql-t5-lora

## Step 6 — Execution-accuracy evaluation
Runs predicted vs. gold SQL against the real DBs and writes the number + samples to `docs/EVAL.md`. Drop `--limit` for the full dev set (slower).

In [ ]:
!python scripts/evaluate_spider.py \
  --base-model "{BASE_MODEL}" --adapter ./nl2sql-t5-lora \
  --tables-json "{TABLES_JSON}" \
  --spider-db-dir "{DB_DIR}" \
  --limit 200

print("\n===== docs/EVAL.md =====")
print(open("docs/EVAL.md").read())

## Step 7 — Push the merged model to the Hugging Face Hub
Merges the saved adapter into the base (no retraining) and uploads. Paste your **write** token when prompted.

In [ ]:
from getpass import getpass
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

login(token=getpass("HF write token: "))

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
merged = PeftModel.from_pretrained(base, "./nl2sql-t5-lora").merge_and_unload()
merged.push_to_hub(MODEL_REPO)
tok.push_to_hub(MODEL_REPO)
print("\nPushed merged model to:", MODEL_REPO)

## Done 🎉

Next:
- Point serving at your model: set the default in `src/nl2sql_agent.py` / `config.py` and the HF Space `MODEL_ID` to **`MODEL_REPO`**.
- Commit the generated `docs/EVAL.md` and update your README/resume bullet with the **measured** execution accuracy.

Tip: download the trained adapter so you don't lose it when the runtime recycles:
```python
from google.colab import files; import shutil
shutil.make_archive('nl2sql-t5-lora', 'zip', 'nl2sql-t5-lora'); files.download('nl2sql-t5-lora.zip')
```